# 08 · Verification

This notebook replicates the Verification paragraph of the report's Results section.

> Verification: Every case-export number reproduced exactly, and on the full database 65 of 66 checks passed; the one failure was a sort-order bug in our comparison script. Independent rebuilds matched the 113 chronic outliers, the fleet rates, all 33,515 compared values of the association study, the follow-ups' headline numbers, and every sheet of the case list. Reviews found that the chronic-outlier count runs from 62 to 133 with the statistical cut-off. The association results' 45 review points were all upheld and applied, and no headline result changed.

**How it works.** Where the evidence is a table, we recompute the claim from it: we count the case-export reproductions' compared values and the database run's checks by status, redo the failing comparison in the corrected order, recount the chronic outliers at three cut-offs from each device's tickets and expected tickets, and compare the association study's headline results with the values pinned before its review. Where the evidence is a record of a past check (the independent rebuilds and the association review), we say so and quote the record from the committed document or commit message that holds it, because the rebuilds' and reviews' own code is not in the repository.

**Re-run:** `reproduce_sql.py`, `reproduce_cleanroom.py`, and `01_excel_reproducibility.py` (the three case-export reproductions, a few seconds each), `high_ticket_machines.py` (about 20 s), and `association.py` (about 70 s). Two re-created files can differ from the committed ones without changing any number here: `repro_sql_claims.csv`, whose `sql_verbatim` column records that the verbatim check was skipped when Zac's notebook file is absent, and `assoc/as_summary.json`, which gains a privacy scan against real identifiers that runs only when IPA's files are present. `01_excel_reproducibility.py` ends by printing its one mismatch, the workbook's checksum.

**Used as committed, without re-running:**
- the record of the full-database run (`db/db_summary.json`, `db/db_pilot_last_distinct_codes.csv`, and `db/db_pilot_last_codes_head_vs_expected.csv`), because `verify_with_db.py` needs the telemetry database, which the notebooks do not re-read; for the same reason, `high_ticket_machines.py` and `association.py` re-run on the committed telemetry-database extracts in `fu/` and `db/` (from `db_followup.py` and `verify_with_db.py`);
- `followups/ag_tests.csv`, from `followups.py`, which this notebook does not re-run;
- the records in `docs/02_verification_report.md`, `docs/05_high_ticket_machines.md`, `docs/06_fleet_exposure.md`, `docs/07_association_protocol.md`, `docs/08_association_results.md`, and `docs/10_followups_results.md`;
- from the repository's history, read with git: the messages of commits cb7ff55 (the case list, Nov 8th) and eeee532 (the association review, Oct 22nd), and the association tables at commit 472f27b (Oct 22nd, before the review).

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    R.rerun(OUT, 'reproduce_sql.py')
    R.rerun(OUT, 'reproduce_cleanroom.py')
    R.rerun(OUT, '01_excel_reproducibility.py')
    R.rerun(OUT, 'high_ticket_machines.py')
    R.rerun(OUT, 'association.py')
R.compare(OUT)

reproduce_sql.py: exit status 0, 5 s
    PASS: every case-only claim reproduces; only the known workbook checksum differs.


reproduce_cleanroom.py: exit status 0, 2 s


01_excel_reproducibility.py: exit status 0, 2 s


high_ticket_machines.py: exit status 0, 18 s
    Done in 17.0 s; checks {'PASS': 33, 'FAIL': 0}


association.py: exit status 0, 103 s
    Done in 101.8 s: 154 checks passed, 0 failed.
73 files written: 71 identical, 2 DIFFERENT.


,file written by the re-run,compared with the committed file
0,01_excel_reproducibility.csv,identical
1,01_pilot_strata.csv,identical
2,01_taxonomy_buckets.csv,identical
3,assoc/as_by_model.csv,identical
4,assoc/as_by_type.csv,identical
...,...,...
68,repro_sql_record_quality.csv,identical
69,repro_sql_repeat_by_model.csv,identical
70,repro_sql_repeat_summary.csv,identical
71,repro_sql_service_groups.csv,identical


## Every case-export number reproduced exactly

Three scripts recompute the midterm analysis's case-export numbers from IPA's case export and write one row per compared value with its status: `reproduce_sql.py` runs Zac's SQL verbatim on the rebuilt case table (`repro_sql_claims.csv`), `reproduce_cleanroom.py` re-derives the numbers in pandas from his method text (`repro_clean_claims.csv`), and `01_excel_reproducibility.py` follows his notebook's sections (`01_excel_reproducibility.csv`); a value matches when it equals Zac's to the digits he printed. We count the rows by status and set aside the workbook's checksum, which differs from that of the copy Zac verified (the local copy was re-saved in Excel on Oct 10th; docs/02 §3), and the values that need the telemetry database or that Zac did not print.

In [3]:
implementations = {  # one row per compared value, with the status its script recorded
    "Zac's SQL, verbatim (reproduce_sql.py)":
        R.read_csv(OUT, 'repro_sql_claims.csv').rename(columns={'claim_id': 'item'}),
    'Clean-room pandas (reproduce_cleanroom.py)':
        R.read_csv(OUT, 'repro_clean_claims.csv').rename(columns={'claim_id': 'item'}),
    "Notebook's sections (01_excel_reproducibility.py)":
        R.read_csv(OUT, '01_excel_reproducibility.csv').rename(columns={'metric': 'item'}).assign(
            status=lambda t: t.match.map({'yes': 'match', 'NO': 'mismatch'})),
}
rows = {}
for name, t in implementations.items():
    checksum = t.item.str.contains('sha256')  # the workbook file's checksum, not a number in the export
    compared = t.status.isin(['match', 'mismatch']) & ~checksum
    aside = ~t.status.isin(['match', 'mismatch']) & ~checksum  # needs the telemetry database, or Zac printed no value
    rows[name] = {'rows': len(t), 'case-export numbers compared': int(compared.sum()),
                  'match': int((compared & t.status.eq('match')).sum()),
                  'differ': int((compared & t.status.eq('mismatch')).sum()),
                  'workbook checksum': ', '.join(t.status[checksum]), 'set aside': int(aside.sum())}
    if aside.any():
        print(f'Set aside in {name}: ' + ', '.join(f'{i} ({s})' for i, s in zip(t.item[aside], t.status[aside])))
case_export = pd.DataFrame(rows).T
n_differ = int(case_export['differ'].sum())
every_label = ('Every' if n_differ == 0 and (case_export['match'] == case_export['case-export numbers compared']).all()
               else f'All but {n_differ}')
print('Case-export numbers that match, by implementation: '
      + '; '.join(f'{m} of {c}' for m, c in zip(case_export['match'], case_export['case-export numbers compared']))
      + f'. Case-export numbers that differ: {n_differ}.')
case_export

Set aside in Zac's SQL, verbatim (reproduce_sql.py): C11_linkage (not_reproducible), C08_pilot_selection (partial), C11_linkage_unmatched_1358 (not_reproducible), C27_reason_split (not_reproducible)
Set aside in Clean-room pandas (reproduce_cleanroom.py): 2.1.customers_for_50pct (not_reproducible), 3.3.type_table_tickets_sum (partial)
Case-export numbers that match, by implementation: 30 of 30; 220 of 220; 47 of 47. Case-export numbers that differ: 0.


,rows,case-export numbers compared,match,differ,workbook checksum,set aside
"Zac's SQL, verbatim (reproduce_sql.py)",35,30,30,0,mismatch,4
Clean-room pandas (reproduce_cleanroom.py),223,220,220,0,not_reproducible,2
Notebook's sections (01_excel_reproducibility.py),48,47,47,0,mismatch,0


## On the full database, 65 of 66 checks passed

`verify_with_db.py` needs the telemetry database, so we read the committed record of its Oct 12th run, `db/db_summary.json`, which lists each check with its expected value, the run's value, and a status. We count the checks that have an expected value (Zac's printed value or an integrity condition) by status, while the INFO rows record a finding with nothing to compare. Two of the run's checks show that it read the database file Zac verified, by its SHA-256, and all of its transactions.

In [4]:
db = R.read_json(OUT, 'db/db_summary.json')  # the committed record of the Oct 12th run of verify_with_db.py
checks = pd.DataFrame(db['checks'])  # one row per check: step, check, expected, actual, status
judged = checks[checks.status.isin(['PASS', 'FAIL'])]  # the checks with an expected value
assert judged.expected.notna().all() and checks.loc[checks.status.eq('INFO'), 'expected'].isna().all()
assert checks.status.value_counts().to_dict() == db['check_counts']
n_checks = len(judged)
n_pass = int(judged.status.eq('PASS').sum())
n_fail = int(judged.status.eq('FAIL').sum())
check_row = checks.set_index('check')
sha = check_row.loc["database SHA-256 equals Zac's verified checksum"]
transactions = check_row.loc['atomic transactions represented']
db_label = 'full database' if sha.status == 'PASS' and transactions.status == 'PASS' else 'not the full database'
print(f'{n_pass} of the {n_checks} checks with an expected value pass and {n_fail} fails; '
      f'{int(checks.status.eq("INFO").sum())} INFO rows record a finding with nothing to compare.')
print(f"The run read the database file Zac verified (SHA-256 check: {sha.status}) and all {transactions.actual:,} "
      f"of its transactions (check: {transactions.status}).")
steps = checks.step.map(lambda s: f"{s}: {db['steps'][s]['title']}")
pd.crosstab(steps, checks.status, margins=True, margins_name='all').rename_axis(index='step', columns=None)

65 of the 66 checks with an expected value pass and 1 fails; 6 INFO rows record a finding with nothing to compare.
The run read the database file Zac verified (SHA-256 check: PASS) and all 118,047,569 of its transactions (check: PASS).


,FAIL,INFO,PASS,all
step,,,,
"0: schema, indexes, formats, counts",0,1,2,3
1: device dictionary,0,0,2,2
2: ticket-device linkage,0,0,11,11
"3: model/type map, mapping summary, host-level audit",0,0,5,5
"4: Zac's aggregation, reconciliation and rate tables",0,1,37,38
4b: 30-ticket pilot reproduction,1,0,5,6
5: clock test: hour profiles by zone x DST period,0,1,0,1
6: clock test: DST transition days,0,1,0,1
7: differing clock rows and ingestion offsets,0,1,2,3


## The one failure was a sort-order bug in our comparison script

The failing check compares the first 15 rows of the run's table of last codes before the pilot tickets with the 15 rows Zac printed, which his notebook ordered by ticket and recency rank. The record, `db/db_pilot_last_codes_head_vs_expected.csv`, took the first 15 rows in the order the run saved them, so we sort the run's committed table, `db/db_pilot_last_distinct_codes.csv`, as Zac did and match his rows again on ticket and rank. docs/02 records the fix in our comparison script.

In [5]:
import re


def flat(text):
    """A document or commit message as one line: single spaces, without quote marks, bold markers, or list dashes."""
    text = ' '.join(line.lstrip('> ') for line in text.splitlines())
    return ' '.join(text.replace('**', '').split()).replace(' - ', ' ')


def quote(doc, pattern):
    """A record in a committed document (path from the repository root): prints the text that matches the pattern
    and returns the match."""
    m = re.search(pattern, flat((R.ROOT / doc).read_text(encoding='utf-8')))
    if m is None:
        raise ValueError(f'{doc}: no text matches {pattern!r}')
    print(f'{doc}: "{m.group(0)}"')
    return m


recorded = R.read_csv(OUT, 'db/db_pilot_last_codes_head_vs_expected.csv', dtype=str)  # Zac's 15 rows, as compared
run_codes = R.read_csv(OUT, 'db/db_pilot_last_distinct_codes.csv', dtype=str)  # the run's table, in its saved order
keys, values = ['case_id', 'recency_rank'], ['transaction_code', 'last_seen', 'occurrences']


def reproduced(first15):
    """Zac's rows that the given rows of the run's table reproduce, matched on ticket and recency rank."""
    both = recorded.merge(first15[keys + values], on=keys, how='left')
    return int(pd.concat([both[v] == both[f'{v}_expected'] for v in values], axis=1).all(axis=1).sum())


as_saved = run_codes.head(15)
resorted = run_codes.assign(rank=run_codes.recency_rank.astype(int)).sort_values(
    ['case_id', 'rank'], kind='mergesort').head(15)
n_recorded = int(recorded.match.eq('True').sum())
n_as_saved, n_sorted = reproduced(as_saved), reproduced(resorted)
assert check_row.loc['pilot_last_codes_head: printed rows reproduced', 'actual'] == f'{n_as_saved}/{len(recorded)}'
sort_label = ('sort-order bug' if n_recorded == n_as_saved == 0 and n_sorted == len(recorded)
              else 'not explained by the order')
fix = quote('docs/02_verification_report.md', r'The one FAIL \(`pilot_last_codes_head`\) was a sort-order bug in our '
            r'comparison script, fixed in commit (\w+); re-sorted, all (\d+) rows match\.')
print(f"Zac's rows reproduced: {n_recorded} of {len(recorded)} in the record vs. {n_sorted} of {len(recorded)} once "
      f"the run's table is sorted by ticket and recency rank.")
pd.DataFrame({'tickets in the first 15 rows': [', '.join(dict.fromkeys(t.case_id)) for t in (as_saved, resorted)],
              "Zac's 15 rows reproduced": [n_as_saved, n_sorted]},
             index=['as the run saved it (the record)', 'sorted by ticket and recency rank (the fix)'])

docs/02_verification_report.md: "The one FAIL (`pilot_last_codes_head`) was a sort-order bug in our comparison script, fixed in commit 1db0943; re-sorted, all 15 rows match."
Zac's rows reproduced: 0 of 15 in the record vs. 15 of 15 once the run's table is sorted by ticket and recency rank.


,tickets in the first 15 rows,Zac's 15 rows reproduced
as the run saved it (the record),"T06663, T03075, T06107",0
sorted by ticket and recency rank (the fix),"T00705, T00783, T00951",15


## Independent rebuilds matched the 113 chronic outliers

A chronic outlier is a device whose machine is in the top 5% by unplanned tickets (`htm_machine_counts.csv`) and whose tickets exceed its telemetry type's rate × its months active beyond chance (one-sided exact Poisson test, Benjamini–Hochberg q < 0.05). We recompute each device's expected tickets, p-value, and q-value from its tickets and months active in `htm_device_exposure.csv` and check that the flagged machines are the ones the script flags. The rebuild itself is a record in docs/05: an independent clean-room build reproduced the same 113 IDs.

In [6]:
import math

import numpy as np


def poisson_sf(k, mu):
    """P(X >= k) for X ~ Poisson(mu), summed over the upper tail as high_ticket_machines.py does (exact for tiny p)."""
    k = int(k)
    if k <= 0:
        return 1.0
    if mu <= 0:
        return 0.0
    if k <= mu:
        return max(0.0, 1.0 - sum(math.exp(-mu + j * math.log(mu) - math.lgamma(j + 1)) for j in range(k)))
    term = math.exp(-mu + k * math.log(mu) - math.lgamma(k + 1))
    total, j = term, k
    while term > total * 1e-17 and j < k + 10_000:
        j += 1
        term *= mu / j
        total += term
    return min(1.0, total)


def bh(p):
    """Benjamini–Hochberg q-values (monotone, capped at 1)."""
    p = np.asarray(p, float)
    order = np.argsort(p, kind='mergesort')
    q = np.minimum.accumulate((p[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    out = np.empty(len(p))
    out[order] = np.minimum(q, 1.0)
    return out


devices = R.read_csv(OUT, 'htm_device_exposure.csv')  # one row per telemetry device in the exposure population
machines = R.read_csv(OUT, 'htm_machine_counts.csv')  # one row per machine (serial) with an unplanned ticket

rank_k = math.ceil(0.05 * len(machines))  # top 5%: at or above the count of the machine at this rank, ties included
cut = int(machines.unplanned_tickets.sort_values(ascending=False).iloc[rank_k - 1])
top5 = devices.serial_id.isin(machines.machine_id[machines.unplanned_tickets >= cut])

group = devices.groupby('rate_group')[['tickets_obs', 'months']].sum()  # each type's tickets per month active
rate = (group.tickets_obs / group.months).where(group.months > 0, devices.tickets_obs.sum() / devices.months.sum())
expected = devices.rate_group.map(rate) * devices.months
tested = expected > 0  # the script tests the devices with expected tickets above 0
p = pd.Series(np.nan, index=devices.index)
p[tested] = [poisson_sf(y, e) for y, e in zip(devices.tickets_obs[tested], expected[tested])]
q = pd.Series(np.nan, index=devices.index)
q[tested] = bh(p[tested])
assert np.allclose(q[tested], devices.q_bh[tested], rtol=1e-9, atol=0), 'q-values differ from the script'
chronic = top5 & (q < 0.05)
n_chronic = int(chronic.sum())
same_machines = set(devices.serial_id[chronic]) == set(machines.machine_id[machines.chronic_outlier])
assert same_machines, 'the recomputed chronic outliers are not the machines the script flags'
rebuild = quote('docs/05_high_ticket_machines.md', r'An independent clean-room build reproduced the populations, the '
                r'outlier classes \(the same (\d+) IDs\) and the episode counts\.')
print(f'Top 5%: {cut} or more unplanned tickets (the count at rank {rank_k} of {len(machines):,} machines). '
      f'{int(tested.sum()):,} devices tested, {n_chronic} chronic outliers, the machines htm_machine_counts.csv flags; '
      f"the record's count is {rebuild.group(1)}.")
pd.crosstab(top5.map({True: 'top 5% by unplanned tickets', False: 'other devices'}),
            (q < 0.05).map({True: 'rate outlier (q < 0.05)', False: 'not a rate outlier'}),
            rownames=[None], colnames=[None], margins=True, margins_name='all').loc[
    ['top 5% by unplanned tickets', 'other devices', 'all'], ['rate outlier (q < 0.05)', 'not a rate outlier', 'all']]

docs/05_high_ticket_machines.md: "An independent clean-room build reproduced the populations, the outlier classes (the same 113 IDs) and the episode counts."
Top 5%: 5 or more unplanned tickets (the count at rank 185 of 3,685 machines). 6,423 devices tested, 113 chronic outliers, the machines htm_machine_counts.csv flags; the record's count is 113.


,rate outlier (q < 0.05),not a rate outlier,all
top 5% by unplanned tickets,113,93,206
other devices,5,6410,6415
all,118,6503,6621


## Independent rebuilds matched the fleet rates

docs/06 records that two clean-room builds, written from the definitions without reading `fleet_exposure.py`, reproduced every number, including the fleet totals and all model and age rates. This is a record of a past check, so we quote it; notebook 04 recomputes the rates themselves from the committed tables.

In [7]:
fleet_record = [
    quote('docs/06_fleet_exposure.md', r'Two clean-room builds, written from the definitions without reading the script, '
          r'reproduced every number\.'),
    quote('docs/06_fleet_exposure.md', r'The reproduction covers: [^.]*all model and age rates[^.]*\.'),
]
fleet_label = 'matched' if all(fleet_record) else 'no record'

docs/06_fleet_exposure.md: "Two clean-room builds, written from the definitions without reading the script, reproduced every number."
docs/06_fleet_exposure.md: "The reproduction covers: the fleet totals; all model and age rates; every customer's observed and expected tickets; all 12 method variants; the consistent-outlier lists; the persistence statistics."


## Independent rebuilds matched all 33,515 compared values of the association study

docs/07 §14 records that two blind clean-room builds of the association study, written from docs/07 alone, matched `association.py` in every compared cell while its results were still hidden. The comparison script is not in the repository, so we quote the record and check the parts of it that the committed tables show: the paired tickets in `assoc/as_pairs.csv` and the rows of `assoc/as_item2_breakdowns.csv`.

In [8]:
blind = quote('docs/07_association_protocol.md',
              r'With those readings, both builds matched `association\.py` in every compared cell, all exactly; none '
              r'needed the Monte Carlo '
              r'tolerances\. Each comparison covered ([\d,]+) cells: the pairs \(([\d,]+) tickets × (\d+) fields\);'
              r'[^.]* (\d+) breakdown rows;[^.]*\.')
n_cells = int(blind.group(1).replace(',', ''))
cells_label = 'all' if 'in every compared cell, all exactly' in blind.group(0) else 'not all'
pairs = R.read_csv(OUT, 'assoc/as_pairs.csv')  # one row per paired ticket
breakdowns = R.read_csv(OUT, 'assoc/as_item2_breakdowns.csv')
assert len(pairs) == int(blind.group(2).replace(',', '')) and len(breakdowns) == int(blind.group(4))
print(f'In the committed tables: {len(pairs):,} paired tickets in as_pairs.csv and {len(breakdowns)} rows in '
      f'as_item2_breakdowns.csv, as the record lists them.')

docs/07_association_protocol.md: "With those readings, both builds matched `association.py` in every compared cell, all exactly; none needed the Monte Carlo tolerances. Each comparison covered 33,515 cells: the pairs (3,259 tickets × 8 fields); the item 2 bucket and code tables, including bootstrap intervals and BH q-values; 860 breakdown rows; H2a for every set and pooled, including intervals and permutation p-values; H1; the item 3 curves."
In the committed tables: 3,259 paired tickets in as_pairs.csv and 860 rows in as_item2_breakdowns.csv, as the record lists them.


## Independent rebuilds matched the follow-ups' headline numbers

docs/10 records that three clean-room builds, written from docs/09 alone, matched `followups.py`'s headline numbers: Part A's global statistic, the gate's outcome, the decomposition, and the halves, and Part B's pairs, classes, directions, and same-by-gap shares. We quote the record and read the global statistic and the gate's outcome from `followups/ag_tests.csv`.

In [9]:
followups_record = quote('docs/10_followups_results.md',
                         r"Three builds, written from docs/09 alone without seeing the script, matched: Part A's global "
                         r"statistic exactly \(X = ([\d.]+)\), the gate's outcome, the decomposition to the ticket, and "
                         r"the halves; Part B's pairs, classes, directions and same-by-gap shares exactly; the null rates "
                         r"within simulation error\.")
primary = R.read_csv(OUT, 'followups/ag_tests.csv').set_index('variant').loc['primary']
x_committed = R.num(primary.statistic_X, 3)
followups_label = 'matched' if x_committed == followups_record.group(1) else 'differs from the record'
print(f'followups/ag_tests.csv: global statistic X = {x_committed} (the record: {followups_record.group(1)}); the gate '
      f"{'passes' if str(primary.global_pass) == 'True' else 'fails'}, and the groups that pass are "
      f"{' and '.join(str(primary.groups_significant).split('; '))}.")

docs/10_followups_results.md: "Three builds, written from docs/09 alone without seeing the script, matched: Part A's global statistic exactly (X = 33.776), the gate's outcome, the decomposition to the ticket, and the halves; Part B's pairs, classes, directions and same-by-gap shares exactly; the null rates within simulation error."
followups/ag_tests.csv: global statistic X = 33.776 (the record: 33.776); the gate passes, and the groups that pass are Mechanical and Electronics.


## Independent rebuilds matched every sheet of the case list

The Electronics case list is a workbook built locally with IPA's case numbers and notes and never committed, so its check survives only as a record: the message of commit cb7ff55 (Nov 8th), which we read from the repository's history with git. It says that an independent clean-room recount matched every sheet row for row, comparing counts only.

In [10]:
import subprocess


def git(*args):
    """Output of a git command in the repository, whose history holds the commit records."""
    return subprocess.run(['git', '-C', str(R.ROOT), *args], capture_output=True, text=True, check=True).stdout


def commit_quote(sha, pattern):
    """A record in a commit message: prints the text that matches the pattern and returns the match."""
    head, _, message = git('log', '-1', '--date=short', '--format=%h %ad%n%B', sha).partition('\n')
    m = re.search(pattern, flat(message))
    if m is None:
        raise ValueError(f'commit {sha}: no text matches {pattern!r}')
    print(f'commit {head}: "{m.group(0)}"')
    return m


recount = commit_quote('cb7ff55', r'An independent clean-room recount matched (every sheet) row for row \(counts only\); '
                       r'the real build passes (\d+) checks and the pseudonymous copy (\d+)\.')
case_list_label = recount.group(1)

commit cb7ff55 2026-11-08: "An independent clean-room recount matched every sheet row for row (counts only); the real build passes 37 checks and the pseudonymous copy 36."


## Reviews found that the chronic-outlier count runs from 62 to 133 with the statistical cut-off

The review in docs/05 §B recounted the chronic outliers at a stricter and a looser Benjamini–Hochberg cut-off, q < 0.01 and q < 0.10, in scratch code that was not committed. We recount them from the q-values recomputed above, with the top-5% cut held fixed, and compare the counts with the review's in docs/05.

In [11]:
review = quote('docs/05_high_ticket_machines.md', r'The count depends on the cut-off \(review\): q < 0\.01 gives (\d+), '
               r'q < 0\.10 gives (\d+), and a top-2\.5% workload cut-off gives \d+\.')
in_docs = {0.01: int(review.group(1)), 0.05: int(rebuild.group(1)), 0.10: int(review.group(2))}
by_cut = pd.DataFrame({f'q < {c:.2f}': {'rate outliers': int((q < c).sum()),
                                         'chronic outliers (top 5% and rate outlier)': int((top5 & (q < c)).sum()),
                                         'chronic outliers in docs/05': n} for c, n in in_docs.items()}).T
n_q01 = int(by_cut.loc['q < 0.01', 'chronic outliers (top 5% and rate outlier)'])
n_q10 = int(by_cut.loc['q < 0.10', 'chronic outliers (top 5% and rate outlier)'])
print(f'The chronic-outlier count runs from {n_q01} at q < 0.01 to {n_q10} at q < 0.10, with {n_chronic} at the '
      f'q < 0.05 used throughout.')
by_cut

docs/05_high_ticket_machines.md: "The count depends on the cut-off (review): q < 0.01 gives 62, q < 0.10 gives 133, and a top-2.5% workload cut-off gives 104."
The chronic-outlier count runs from 62 at q < 0.01 to 133 at q < 0.10, with 113 at the q < 0.05 used throughout.


,rate outliers,chronic outliers (top 5% and rate outlier),chronic outliers in docs/05
q < 0.01,65,62,62
q < 0.05,118,113,113
q < 0.10,145,133,133


## The association results' 45 review points were all upheld and applied

docs/08 records the skeptic review of the association results: five reviewers checked the analysis and traced every number, a second reviewer tried to refute each of the 45 findings and refuted none, and all were applied. We quote the record and the subject of commit eeee532, which applied the findings, and list the association tables that commit changed.

In [12]:
review_record = quote('docs/08_association_results.md', r'A second reviewer then tried to refute each major finding; none '
                      r'of the (\d+) findings was refuted\. All are applied here:')
n_points = int(review_record.group(1))
review_commit = commit_quote('eeee532', r'Apply the item 8 skeptic review \((\d+) findings\) to the association study')
assert int(review_commit.group(1)) == n_points
upheld_label = 'all' if 'none of the' in review_record.group(0) and 'All are applied' in review_record.group(0) else 'not all'
changed = [f.rsplit('/', 1)[-1] for f in git('diff', '--name-only', '472f27b', 'eeee532', '--',
                                              'verification/outputs/assoc').split()]
print(f'Association tables the review commit changed ({len(changed)}): {", ".join(changed)}.')

docs/08_association_results.md: "A second reviewer then tried to refute each major finding; none of the 45 findings was refuted. All are applied here:"
commit eeee532 2026-10-22: "Apply the item 8 skeptic review (45 findings) to the association study"
Association tables the review commit changed (6): as_by_type.csv, as_guard.csv, as_item4.csv, as_match_rates.csv, as_sensitivities.csv, as_summary.json.


## No headline result changed

`association.py` pins its headline results, the pooled specificity ratio (H2a) and H1 at 7 and 30 days with their intervals and p-values or counts, to the values set on Oct 22nd when the outputs were unblinded, before the review. We compare the pins, which `assoc/as_summary.json` lists, with `assoc/as_h2a.csv` and `assoc/as_h1.csv`, and with the same tables at commit 472f27b, which set the pins; the intervals and p-values need the script's bootstrap and permutations, so we compare them as the tables give them.

In [13]:
import io


def headline(h2a, h1):
    """The pinned headline rows of association.py: pooled H2a (log ratio, interval, permutation p) and H1 (ratio,
    interval, b, c) at 7 days (168 h) and 30 days (720 h)."""
    rows = {}
    for k in (168, 720):
        r = h2a[(h2a.k == k) & (h2a.set == 'POOLED')].iloc[0]
        rows[f'pinned pooled H2a at k={k} (estimate, interval, permutation p)'] = [r.estimate, r.ci_lo, r.ci_hi, r.perm_p]
        r = h1[h1.k == k].iloc[0]
        rows[f'pinned H1 at k={k} (estimate, interval, b, c)'] = [r.estimate, r.ci_lo, r.ci_hi, r.sum_b, r.sum_c]
    return rows


def shown(name, v):
    """A headline row as the report reads it: H2a on the ratio scale."""
    if 'H2a' in name:
        return f'{math.exp(v[0]):.2f} ({math.exp(v[1]):.2f}–{math.exp(v[2]):.2f}), p = {v[3]:.4f}'
    return f'{v[0]:.2f} ({v[1]:.2f}–{v[2]:.2f}), b = {v[3]:.0f}, c = {v[4]:.0f}'


def at_commit(sha, name):
    """An association table as committed at an earlier commit, read from the repository's history."""
    return pd.read_csv(io.StringIO(git('show', f'{sha}:verification/outputs/assoc/{name}')))


summary = R.read_json(OUT, 'assoc/as_summary.json')
pins = {c['check']: c['expected'] for c in summary['check_list'] if c['check'].startswith('pinned ')}
now = headline(R.read_csv(OUT, 'assoc/as_h2a.csv'), R.read_csv(OUT, 'assoc/as_h1.csv'))
before = headline(at_commit('472f27b', 'as_h2a.csv'), at_commit('472f27b', 'as_h1.csv'))
assert set(pins) == set(now) == set(before)
same = {name: bool(np.allclose(now[name], pins[name], rtol=0, atol=1e-6) and np.allclose(now[name], before[name]))
        for name in pins}
n_changed = sum(not s for s in same.values())
quote('docs/07_association_protocol.md', r'Effect on headline results: none of these changes any headline result\. '
      r'They are pinned as checks\.')
commit_quote('eeee532', r'(\d+) checks pass; no headline result changed\.')
print(f'Headline results that differ from their pins or from commit 472f27b: {n_changed} of {len(pins)}.')
label = {n: f"{'pooled H2a' if 'H2a' in n else 'H1'}, {7 if 'k=168' in n else 30} days" for n in pins}
pd.DataFrame({'pinned (as_summary.json)': {n: shown(n, v) for n, v in pins.items()},
              'at commit 472f27b': {n: shown(n, v) for n, v in before.items()},
              'committed tables now': {n: shown(n, v) for n, v in now.items()},
              'same': same}).rename(index=label)

docs/07_association_protocol.md: "Effect on headline results: none of these changes any headline result. They are pinned as checks."
commit eeee532 2026-10-22: "153 checks pass; no headline result changed."
Headline results that differ from their pins or from commit 472f27b: 0 of 4.


,pinned (as_summary.json),at commit 472f27b,committed tables now,same
"pooled H2a, 7 days","3.22 (2.34–4.77), p = 0.0005","3.22 (2.34–4.77), p = 0.0005","3.22 (2.34–4.77), p = 0.0005",True
"H1, 7 days","6.48 (4.95–8.87), b = 311, c = 48","6.48 (4.95–8.87), b = 311, c = 48","6.48 (4.95–8.87), b = 311, c = 48",True
"pooled H2a, 30 days","1.90 (1.39–2.75), p = 0.0010","1.90 (1.39–2.75), p = 0.0010","1.90 (1.39–2.75), p = 0.0010",True
"H1, 30 days","4.10 (3.06–5.79), b = 205, c = 50","4.10 (3.06–5.79), b = 205, c = 50","4.10 (3.06–5.79), b = 205, c = 50",True


## Check against the report

In [14]:
number_word = {0: 'no', 1: 'one', 2: 'two', 3: 'three'}
R.check([
    ('Case-export numbers that reproduce in all three implementations', 'Every', every_label),
    ('Database the checks ran on (SHA-256 and transaction-count checks)', 'full database', db_label),
    ('Database run: checks that passed', '65', R.num(n_pass)),
    ('Database run: checks with an expected value', '66', R.num(n_checks)),
    ('Database run: failed checks', 'one', number_word.get(n_fail, R.num(n_fail))),
    ("Cause of the failure (Zac's rows reproduced once sorted)", 'sort-order bug', sort_label),
    ('Chronic outliers (top 5% and q < 0.05)', '113', R.num(n_chronic)),
    ('Fleet rates: two clean-room builds (docs/06 record)', 'matched', fleet_label),
    ('Association study: values compared per clean-room build (docs/07 record)', '33,515', R.num(n_cells)),
    ('Association study: compared values that matched (docs/07 record)', 'all', cells_label),
    ("Follow-ups' headline numbers: three clean-room builds (docs/10 record)", 'matched', followups_label),
    ('Case list: sheets the independent recount matched (commit cb7ff55)', 'every sheet', case_list_label),
    ('Chronic outliers at q < 0.01 (review, docs/05)', '62', R.num(n_q01)),
    ('Chronic outliers at q < 0.10 (review, docs/05)', '133', R.num(n_q10)),
    ('Association results: review points (docs/08 record)', '45', R.num(n_points)),
    ('Review points upheld and applied (docs/08 record)', 'all', upheld_label),
    ('Headline results that changed after the review', 'no', number_word.get(n_changed, R.num(n_changed))),
]);

,Finding,Report,Recomputed,Match
0,Case-export numbers that reproduce in all thre...,Every,Every,yes
1,Database the checks ran on (SHA-256 and transa...,full database,full database,yes
2,Database run: checks that passed,65,65,yes
3,Database run: checks with an expected value,66,66,yes
4,Database run: failed checks,one,one,yes
5,Cause of the failure (Zac's rows reproduced on...,sort-order bug,sort-order bug,yes
6,Chronic outliers (top 5% and q < 0.05),113,113,yes
7,Fleet rates: two clean-room builds (docs/06 re...,matched,matched,yes
8,Association study: values compared per clean-r...,"33,515","33,515",yes
9,Association study: compared values that matche...,all,all,yes


All 17 values match the report.
